# Page Expansion — Is It Worth It?

**Question**: If we expand each retrieved chunk to include all sibling chunks from the same `(doc_name, page_num)`, do we actually pick up content that helps answer queries — or does it just add noise?

**Approach**: Use the live Qdrant store. Three analyses:

1. **Chunk fragmentation** — how often does a single page produce multiple chunks, and how small are the tiny ones (likely headings / icon-only chunks)?
2. **Top-fragmented pages** — show what the chunks on the most-split pages actually contain, to judge whether sibling chunks carry meaning.
3. **Retrieval simulation** — run real queries; show what `retrieve_pages` returns vs. what page expansion would add. Includes the Mission 3 example.

Decision criterion: if a meaningful fraction of retrieved top-k chunks have unretrieved siblings that contain content a human would want when answering the question, page expansion is worth implementing.

## Setup

In [ ]:
import sys
from collections import defaultdict, Counter
from pathlib import Path

# Make the package importable from the notebook
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qdrant_client import models

from boardgame_agent.config import COLLECTION_NAME
from boardgame_agent.rag.indexer import get_qdrant_client
from boardgame_agent.rag.retriever import retrieve_pages, format_pages_for_llm

client = get_qdrant_client()
print(f"Collection: {COLLECTION_NAME}")
print(f"Exists: {client.collection_exists(COLLECTION_NAME)}")

In [ ]:
# ── Choose a game to analyze ──────────────────────────────
GAME_ID = "the_crew__the_quest_for_planet_nine"
# ──────────────────────────────────────────────────────────

# Show available game_ids in the collection so it's easy to switch.
scroll_iter = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10_000,
    with_payload=["game_id"],
    with_vectors=False,
)
all_game_ids = Counter(p.payload["game_id"] for p in scroll_iter[0])
print("game_id -> chunk count")
for gid, n in sorted(all_game_ids.items(), key=lambda x: -x[1]):
    marker = " ← selected" if gid == GAME_ID else ""
    print(f"  {gid:50s} {n}{marker}")

In [ ]:
# Fetch every chunk for the selected game (full payloads, no vectors)
def fetch_all_chunks_for_game(game_id: str) -> list[dict]:
    chunks = []
    offset = None
    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=models.Filter(
                must=[models.FieldCondition(key="game_id", match=models.MatchValue(value=game_id))]
            ),
            limit=512,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        chunks.extend({"id": p.id, **p.payload} for p in points)
        if offset is None:
            break
    return chunks

chunks = fetch_all_chunks_for_game(GAME_ID)
print(f"Total chunks for {GAME_ID}: {len(chunks)}")
print(f"Documents: {sorted(set(c['doc_name'] for c in chunks))}")
print(f"Doc tags:  {sorted(set(c.get('doc_tag', '(none)') for c in chunks))}")

## 1. Chunk fragmentation per page

How many chunks does each page produce? A page that produces 1 chunk needs no expansion. A page that produces 3+ chunks is where expansion might help.

In [ ]:
# Group chunks by (doc_name, page_num)
page_groups: dict[tuple[str, int], list[dict]] = defaultdict(list)
for c in chunks:
    page_groups[(c["doc_name"], c["page_num"])].append(c)

chunks_per_page = Counter(len(v) for v in page_groups.values())

print(f"Total unique pages: {len(page_groups)}")
print()
print("Distribution of chunks-per-page:")
print(f"{'chunks':>6s}  {'pages':>6s}  {'% pages':>8s}")
total_pages = sum(chunks_per_page.values())
for n in sorted(chunks_per_page):
    pages = chunks_per_page[n]
    pct = 100 * pages / total_pages
    bar = "█" * int(pct / 2)
    print(f"{n:>6d}  {pages:>6d}  {pct:>6.1f}%  {bar}")

multi_chunk_pages = sum(v for k, v in chunks_per_page.items() if k > 1)
print(f"\nPages with >1 chunk: {multi_chunk_pages} / {total_pages} ({100*multi_chunk_pages/total_pages:.1f}%)")
print("→ Page expansion only matters for these pages. If this number is low, the change is low-value.")

In [ ]:
# How small are individual chunks? Tiny ones (single-bbox or very short text)
# are the strongest case for expansion — they're the ones most likely to be
# a stranded heading, caption, or icon row.
import statistics

chunk_text_lens = [len(c.get("text", "")) for c in chunks]
bbox_counts = [len(c.get("bboxes", [])) for c in chunks]
picture_bbox_counts = [
    sum(1 for b in c.get("bboxes", []) if b.get("label") == "picture") for c in chunks
]

print("Chunk text length (chars):")
print(f"  min/median/mean/max: {min(chunk_text_lens)} / {int(statistics.median(chunk_text_lens))} / {int(statistics.mean(chunk_text_lens))} / {max(chunk_text_lens)}")
print()
print("Bboxes per chunk:")
print(f"  min/median/mean/max: {min(bbox_counts)} / {int(statistics.median(bbox_counts))} / {statistics.mean(bbox_counts):.1f} / {max(bbox_counts)}")
print()

tiny = [c for c in chunks if len(c.get("text", "")) < 100]
single_bbox = [c for c in chunks if len(c.get("bboxes", [])) <= 1]
picture_only = [
    c for c in chunks
    if c.get("bboxes") and all(b.get("label") == "picture" for b in c["bboxes"])
]

print(f"Tiny chunks (<100 chars):       {len(tiny):>4d} / {len(chunks)} ({100*len(tiny)/len(chunks):.1f}%)")
print(f"Single-bbox chunks:             {len(single_bbox):>4d} / {len(chunks)} ({100*len(single_bbox)/len(chunks):.1f}%)")
print(f"Picture-only chunks:            {len(picture_only):>4d} / {len(chunks)} ({100*len(picture_only)/len(chunks):.1f}%)")
print()
print("→ Tiny / single-bbox / picture-only chunks are the strongest case for expansion:")
print("   on their own they retrieve a fragment; with siblings attached they retrieve")
print("   the surrounding section.")

## 2. Top-fragmented pages — what's actually on them?

Show the pages with the most chunks. Scan the chunk text to judge: do these chunks belong together (expansion helps), or are they genuinely independent sub-sections (expansion adds noise)?

In [ ]:
TOP_N_PAGES = 8
TEXT_PREVIEW_CHARS = 180

most_fragmented = sorted(page_groups.items(), key=lambda kv: -len(kv[1]))[:TOP_N_PAGES]

for (doc_name, page_num), page_chunks in most_fragmented:
    print("=" * 80)
    print(f"DOC: {doc_name}  PAGE: {page_num}  CHUNKS: {len(page_chunks)}")
    print("=" * 80)
    for i, c in enumerate(page_chunks):
        text = c.get("text", "")
        preview = text[:TEXT_PREVIEW_CHARS].replace("\n", " ")
        n_bbox = len(c.get("bboxes", []))
        n_pic = sum(1 for b in c.get("bboxes", []) if b.get("label") == "picture")
        labels = Counter(b.get("label", "?") for b in c.get("bboxes", []))
        label_str = ", ".join(f"{k}={v}" for k, v in labels.items())
        print(f"\n  [{i}] {len(text):>4d} chars  |  bboxes: {n_bbox} ({label_str})  |  pics: {n_pic}")
        print(f"      {preview!r}")
    print()

## 3. Retrieval simulation — what does expansion add for real queries?

Run the existing `retrieve_pages` for a few sample queries. For each top-k hit, fetch its sibling chunks (same `doc_name + page_num`) that weren't already returned. Show the gained content so you can judge whether it's meaningful or noise.

In [ ]:
def fetch_siblings(doc_name: str, page_num: int, exclude_ids: set) -> list[dict]:
    """Return all chunks on (doc_name, page_num) NOT already in exclude_ids."""
    points, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(key="game_id", match=models.MatchValue(value=GAME_ID)),
                models.FieldCondition(key="doc_name", match=models.MatchValue(value=doc_name)),
                models.FieldCondition(key="page_num", match=models.MatchValue(value=page_num)),
            ]
        ),
        limit=64,
        with_payload=True,
        with_vectors=False,
    )
    return [{"id": p.id, **p.payload} for p in points if p.id not in exclude_ids]


def show_retrieval_with_expansion(query: str, doc_tag: str | None = None, k: int = 5):
    print("#" * 80)
    print(f"QUERY: {query!r}    (doc_tag={doc_tag!r}, k={k})")
    print("#" * 80)
    points = retrieve_pages(client, query, GAME_ID, k=k, doc_tag=doc_tag)
    retrieved_ids = {p.id for p in points}

    total_sibling_chars = 0
    total_sibling_chunks = 0
    total_sibling_pictures = 0

    for rank, p in enumerate(points, 1):
        pl = p.payload
        doc_name = pl["doc_name"]
        page_num = pl["page_num"]
        text = pl.get("text", "")
        print(f"\n── #{rank}  {doc_name} p.{page_num}  ({len(text)} chars)  doc_tag={pl.get('doc_tag')}")
        print(f"   RETRIEVED CHUNK: {text[:240]!r}")

        siblings = fetch_siblings(doc_name, page_num, retrieved_ids)
        if not siblings:
            print("   (no siblings on this page — expansion would add nothing)")
            continue

        for j, s in enumerate(siblings):
            stext = s.get("text", "")
            n_pic = sum(1 for b in s.get("bboxes", []) if b.get("label") == "picture")
            total_sibling_chars += len(stext)
            total_sibling_chunks += 1
            total_sibling_pictures += n_pic
            print(f"   + SIBLING [{j}] {len(stext)} chars, {n_pic} pictures: {stext[:240]!r}")

    print("\n" + "-" * 80)
    print(f"EXPANSION ADDS: {total_sibling_chunks} chunks, {total_sibling_chars} chars, {total_sibling_pictures} picture bboxes")
    print("-" * 80)

In [ ]:
# Mission 3 — the canonical cross-reference failure case
show_retrieval_with_expansion(
    "Can you explain the criteria for winning on mission 3",
    doc_tag=None,
    k=5,
)

In [ ]:
# Text-only baseline — sibling adds should be modest or zero here
show_retrieval_with_expansion(
    "who starts the game",
    doc_tag=None,
    k=5,
)

In [ ]:
# Icon-meaning question — does the retrieved chunk land you on a page where
# the icon definition is a sibling chunk that wouldn't otherwise be returned?
show_retrieval_with_expansion(
    "what does the red starburst symbol mean on mission cards",
    doc_tag=None,
    k=5,
)

In [ ]:
# Another multi-source mission question to test pattern consistency
show_retrieval_with_expansion(
    "how do you win mission 12",
    doc_tag=None,
    k=5,
)

## 4. Aggregate signal across many queries

Run a batch of queries and quantify: across all top-k results, what fraction have at least one sibling chunk that would be added by expansion? What's the average sibling-chars-added per retrieval?

**Edit `QUERIES` to test your own corpus.**

In [ ]:
QUERIES = [
    "who starts the game",
    "how do you win mission 3",
    "how do you win mission 12",
    "what is the commander",
    "what does the red starburst symbol mean",
    "how does communication work",
    "setup for 4 players",
    "what is a trick",
    "distress signal rules",
    "task tokens",
]

results = []
for q in QUERIES:
    points = retrieve_pages(client, q, GAME_ID, k=5, doc_tag=None)
    retrieved_ids = {p.id for p in points}
    for p in points:
        pl = p.payload
        siblings = fetch_siblings(pl["doc_name"], pl["page_num"], retrieved_ids)
        sibling_chars = sum(len(s.get("text", "")) for s in siblings)
        sibling_pictures = sum(
            sum(1 for b in s.get("bboxes", []) if b.get("label") == "picture")
            for s in siblings
        )
        results.append({
            "query": q,
            "doc": pl["doc_name"],
            "page": pl["page_num"],
            "chunk_chars": len(pl.get("text", "")),
            "n_siblings": len(siblings),
            "sibling_chars": sibling_chars,
            "sibling_pictures": sibling_pictures,
        })

n_with_siblings = sum(1 for r in results if r["n_siblings"] > 0)
n_with_picture_siblings = sum(1 for r in results if r["sibling_pictures"] > 0)

print(f"Total (query, top-k-hit) pairs:                 {len(results)}")
print(f"  with ≥1 sibling chunk:                        {n_with_siblings} ({100*n_with_siblings/len(results):.0f}%)")
print(f"  with ≥1 sibling picture-bbox:                 {n_with_picture_siblings} ({100*n_with_picture_siblings/len(results):.0f}%)")
print()
avg_sibling_chars = sum(r["sibling_chars"] for r in results) / len(results)
print(f"Avg sibling chars added per retrieved chunk:    {avg_sibling_chars:.0f}")
print()
print("Per-query breakdown:")
print(f"{'query':<50s}  {'hits w/siblings':>16s}  {'+chars':>8s}  {'+pics':>6s}")
by_query: dict[str, list[dict]] = defaultdict(list)
for r in results:
    by_query[r["query"]].append(r)
for q, rs in by_query.items():
    n = sum(1 for r in rs if r["n_siblings"] > 0)
    chars = sum(r["sibling_chars"] for r in rs)
    pics = sum(r["sibling_pictures"] for r in rs)
    print(f"{q[:48]:<50s}  {n:>5d}/{len(rs):<10d}  {chars:>8d}  {pics:>6d}")

## 5. Verdict

Look at the numbers from sections 1–4 and judge:

- **If §1 shows most pages have 1 chunk** → page expansion is a no-op for most queries. Probably not worth implementing.
- **If §2 shows fragmented pages contain related content split across chunks** (heading + body, flavor text + icon rows) → expansion is likely high-value.
- **If §2 shows fragmented pages contain genuinely independent sub-sections** (different rules, different topics) → expansion will add noise.
- **If §4 shows most retrieved chunks have siblings that include picture-bboxes** → expansion specifically helps the icon-cross-reference failure mode you described.
- **If §4 shows the Mission 3 query already retrieves all relevant chunks (no siblings to add)** → the failure mode is elsewhere (rerank, prompts, decomposition) and page expansion alone won't fix it.

The honest answer comes from the data, not the diagnosis. If this notebook shows expansion adds <10% sibling content on average and the Mission 3 case isn't materially improved, prioritize a different change first.